# Logistic Regression Experiments
This notebook is a lightweight sandbox for primitive test runs.

Run cells in order from top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
import csv

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Data root in this repo
BASE_DIR = "datasets_full.csv/datasets_full.csv"
USERS_FILE = "users.csv"
TWEETS_FILE = "tweets.csv"

DATASETS = {
    "genuine_accounts.csv": 0,
    "fake_followers.csv": 1,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

TEXT_COLUMN = "text"

USER_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TWEET_FEATURES = [
   "reply_count", "favorite_count", "num_urls", "num_mentions"
]

print("Dataset root:", BASE_DIR)
print("Files configured:", len(DATASETS))

Dataset root: datasets_full.csv/datasets_full.csv
Files configured: 9


In [2]:
# Block 1: Data loading (full configured datasets)

TWEET_COLS = ["id", "text", "source", "user_id", "truncated", "in_reply_to_status_id", 
              "in_reply_to_user_id", "in_reply_to_screen_name", "retweeted_status_id", 
              "geo", "place", "contributors", "retweet_count", "reply_count", 
              "like_count", "quote_count", "is_retweet", "retweeted_status_user_id",
              "created_at", "collected_at", "updated_at", "updated_at2"]

def resolve_users_csv(base_dir, dataset_entry):
    """Resolve dataset entry to a users.csv path."""
    candidates = [
        os.path.join(base_dir, dataset_entry),
        os.path.join(base_dir, dataset_entry, USERS_FILE),
        os.path.join(base_dir, dataset_entry, dataset_entry, USERS_FILE),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def resolve_tweets_csv(base_dir, dataset_entry):
    """Resolve dataset entry to a users.csv path."""
    candidates = [
        os.path.join(base_dir, dataset_entry),
        os.path.join(base_dir, dataset_entry, TWEETS_FILE),
        os.path.join(base_dir, dataset_entry, dataset_entry, TWEETS_FILE),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def load_all_data(dataset_map, base_dir):
    """Load all configured datasets and attach binary labels."""
    frames = []
    missing_files = []

    for dataset_entry, label in dataset_map.items():
        path = resolve_users_csv(base_dir, dataset_entry)
        if path is None:
            missing_files.append(dataset_entry)
            continue

        df = pd.read_csv(
            path,
            encoding="utf-8",
            on_bad_lines="skip",
            low_memory=False,
        )
        df["label"] = int(label)
        df["source_file"] = dataset_entry
        frames.append(df)
        print(f"Loaded {dataset_entry}: {len(df):,} rows")

    if missing_files:
        print("\nMissing dataset entries:")
        for p in missing_files:
            print(" -", p)

    if not frames:
        raise ValueError("No dataset files were loaded. Check BASE_DIR and DATASETS.")

    all_data = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows loaded: {len(all_data):,}")
    return all_data


def load_tweets(base_dir, dataset_entry, chunksize=50_000, max_text_chars=500):
    path = resolve_tweets_csv(base_dir, dataset_entry)
    if path is None:
        return None

    try:
        first_row = pd.read_csv(path, nrows=1, header=None, encoding="latin-1",
                                on_bad_lines="skip", engine="python", quoting=csv.QUOTE_NONE)
    except Exception as e:
        print(f"Skipping {dataset_entry}: could not read — {e}")
        return None

    has_header = str(first_row.iloc[0, 0]).strip().lower() == "id"
    num_cols = first_row.shape[1]
    col_names = TWEET_COLS[:num_cols] if not has_header else None
    target_cols = {"id", "user_id", "text"} | set(TWEET_FEATURES)

    agg_cols_acc = {}
    text_acc = {}

    def process_chunks(reader):
        for chunk in reader:
            if "user_id" not in chunk.columns:
                continue
            chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
            agg_cols = [c for c in TWEET_FEATURES if c in chunk.columns]
            if agg_cols:
                chunk[agg_cols] = chunk[agg_cols].apply(pd.to_numeric, errors="coerce")
                for uid, grp in chunk.groupby("user_id"):
                    if uid not in agg_cols_acc:
                        agg_cols_acc[uid] = {c: [] for c in agg_cols}
                    for c in agg_cols:
                        agg_cols_acc[uid][c].extend(grp[c].dropna().tolist())
            if "text" in chunk.columns:
                for uid, grp in chunk.groupby("user_id"):
                    existing = text_acc.get(uid, "")
                    if len(existing) < max_text_chars:
                        new_text = " ".join(grp["text"].dropna().astype(str))
                        text_acc[uid] = (existing + " " + new_text)[:max_text_chars]

    shared_kwargs = dict(
        header=0 if has_header else None,
        names=col_names,
        encoding="latin-1",
        on_bad_lines="skip",
        na_values=["NULL"],
        chunksize=chunksize,
    )

    try:
        reader = pd.read_csv(path, **shared_kwargs, engine="c", low_memory=False,
                             usecols=lambda c: c in target_cols)
        process_chunks(reader)
        print(f"Loaded {dataset_entry} (C engine)")
    except Exception:
        try:
            reader = pd.read_csv(path, **shared_kwargs, engine="python",
                                 quoting=csv.QUOTE_NONE,
                                 usecols=lambda c: c in target_cols)
            process_chunks(reader)
            print(f"Loaded {dataset_entry} (Python engine fallback)")
        except Exception as e:
            print(f"Skipping {dataset_entry}: parse error — {e}")
            return None

    if not agg_cols_acc and not text_acc:
        return None

    all_ids = set(agg_cols_acc) | set(text_acc)
    rows = []
    for uid in all_ids:
        row = {"user_id": uid}
        if uid in agg_cols_acc:
            for c, vals in agg_cols_acc[uid].items():
                row[c] = sum(vals) / len(vals) if vals else float("nan")
        if uid in text_acc:
            row["text"] = text_acc[uid]
        rows.append(row)

    return pd.DataFrame(rows)


raw = load_all_data(DATASETS, BASE_DIR)

tweet_aggs = []
for dataset_entry in DATASETS:
    agg = load_tweets(BASE_DIR, dataset_entry)
    if agg is not None:
        agg["source_file"] = dataset_entry
        tweet_aggs.append(agg)

if tweet_aggs:
    all_tweets = pd.concat(tweet_aggs, ignore_index=True)
    all_tweets["user_id"] = pd.to_numeric(all_tweets["user_id"], errors="coerce")
    raw["id"] = pd.to_numeric(raw["id"], errors="coerce")
    raw = raw.merge(all_tweets, left_on="id", right_on="user_id", how="left")
    print(f"Tweets merged. Rows with text: {raw['text'].notna().sum():,}")


Loaded genuine_accounts.csv: 3,474 rows
Loaded fake_followers.csv: 3,351 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 14,368


/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:91: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  for chunk in reader:
/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:91: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  for chunk in reader:
/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:91: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  for chunk in reader:
/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:91: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  for chunk in reader:
/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:91: P

Loaded genuine_accounts.csv (Python engine fallback)
Loaded fake_followers.csv (C engine)
Loaded social_spambots_1.csv (C engine)
Loaded social_spambots_2.csv (C engine)
Loaded social_spambots_3.csv (C engine)
Loaded traditional_spambots_1.csv (C engine)
Tweets merged. Rows with text: 3,964


/var/folders/fz/zn_qgggn0s1brtkg6_72tn4c0000gn/T/ipykernel_67045/3538762635.py:165: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  raw = raw.merge(all_tweets, left_on="id", right_on="user_id", how="left")


In [3]:
# Block 2: Slicing and preprocessing
available_numeric = [c for c in USER_FEATURES if c in raw.columns]
if not available_numeric:
    raise ValueError("No numeric feature columns found in raw data.")

X_num = raw[available_numeric].copy()
X_num = X_num.apply(pd.to_numeric, errors="coerce")

tweet_features = [c for c in TWEET_FEATURES if c in raw.columns]
X_num = pd.concat([X_num, raw[tweet_features]], axis=1)

X_num = X_num.fillna(0)

text_data = raw[TEXT_COLUMN].fillna("")

tfidf = TfidfVectorizer(
    max_features = 2000, 
    ngram_range = (1, 2),
    stop_words = "english"
)

print(raw["id"].dtype)
print(all_tweets["user_id"].dtype)

print(all_tweets.shape)
print(all_tweets["text"].notna().sum())

for dataset_entry in DATASETS:
    path = resolve_tweets_csv(BASE_DIR, dataset_entry)
    print(dataset_entry, "->", path)

print(raw["text"].isna().sum(), "nulls out of", len(raw))
print(raw["text"].fillna("").str.strip().eq("").sum(), "empty strings")

print(all_tweets.columns.tolist())

X_text = tfidf.fit_transform(text_data)

X = hstack([X_num.values, X_text])
    
# print(X.columns) 
# print(X.shape)

print(raw.columns.tolist())
print(TEXT_COLUMN)

y = raw["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class balance:\n", y.value_counts(dropna=False))

int64
float64
(188412, 4)
188412
genuine_accounts.csv -> datasets_full.csv/datasets_full.csv/genuine_accounts.csv/tweets.csv
fake_followers.csv -> datasets_full.csv/datasets_full.csv/fake_followers.csv/tweets.csv
social_spambots_1.csv -> datasets_full.csv/datasets_full.csv/social_spambots_1.csv/tweets.csv
social_spambots_2.csv -> datasets_full.csv/datasets_full.csv/social_spambots_2.csv/tweets.csv
social_spambots_3.csv -> datasets_full.csv/datasets_full.csv/social_spambots_3.csv/tweets.csv
traditional_spambots_1.csv -> datasets_full.csv/datasets_full.csv/traditional_spambots_1.csv/tweets.csv
traditional_spambots_2.csv -> None
traditional_spambots_3.csv -> None
traditional_spambots_4.csv -> None
10406 nulls out of 14370
10962 empty strings
['user_id', 'reply_count', 'text', 'source_file']
['id', 'name', 'screen_name', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'url', 'lang', 'time_zone', 'location', 'default_profile', 'default_profile_image

In [4]:
# Block 3: Training
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled, y_train)

print("Training complete.")
print("Model intercept:", model.intercept_[0])

Training complete.
Model intercept: 1.276645503996913


In [5]:
# Block 4: Evaluation
pred = model.predict(X_test_scaled)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

coef_df = pd.DataFrame({
    # "feature": X.columns,
    "coefficient": model.coef_[0],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 features by |coefficient|:")
display(coef_df.head(10))

Accuracy: 0.9179

Confusion Matrix:
[[ 576  119]
 [ 117 2062]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8312    0.8288    0.8300       695
           1     0.9454    0.9463    0.9459      2179

    accuracy                         0.9179      2874
   macro avg     0.8883    0.8875    0.8879      2874
weighted avg     0.9178    0.9179    0.9178      2874


Top 10 features by |coefficient|:


,coefficient,abs_coefficient
3,-17.460686,17.460686
0,-1.054718,1.054718
1,1.016488,1.016488
2,0.814496,0.814496
5,-0.772145,0.772145
8,-0.720002,0.720002
4,-0.592674,0.592674
1249,-0.418206,0.418206
1708,-0.296013,0.296013
436,0.253387,0.253387


## Run Order
1. Cell 2: imports and constants.
2. Cell 3: full data loading.
3. Cell 4: slicing and preprocessing.
4. Cell 5: training.
5. Cell 6: evaluation.

## Notes
- This notebook now loads all configured dataset files directly.
- If runtime or memory gets heavy, reduce loaded columns or process in chunks.